# Rebuild AOI-level water-hyacinth time series

This notebook reconstructs the AOI-level water-hyacinth (WH) time series directly from the **final classified GeoTIFFs** produced by `Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb`.

It mirrors the key conventions in the main classifier notebook:

- floating plants / WH class code = **2**
- classification nodata value = **255**
- dates are parsed from `YYYY-MM-DD_to_YYYY-MM-DD` in each filename
- paper-rule rasters (`*_local_rules.tif`) are kept distinct from model rasters
- probability rasters and intermediate tile rasters are ignored
- if both an original S1 model raster and a `*_patch_cleaned.tif` version exist, the patch-cleaned raster is preferred

For each classified raster it calculates WH area and the percentage of valid classified AOI occupied by WH, then saves a tidy CSV and a publication-ready PNG.


In [ ]:
# Colab setup
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("rasterio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


## Configuration

Usually you only need to change `CLASSIFIED_DIR`.

The default path matches the output structure of the full-stack classifier notebook.


In [ ]:
CLASSIFIED_DIR = Path(
    "/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch/classified_geotiffs"
)

OUTPUT_DIR = CLASSIFIED_DIR.parent
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = TABLE_DIR / "winam_rebuilt_aoi_wh_timeseries.csv"
PNG_PATH = FIGURE_DIR / "winam_rebuilt_aoi_wh_timeseries.png"

# Matches Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb
FLOATING_CLASS_CODE = 2
NODATA_VALUE = 255

# Figure controls
PLOT_SENSOR = "S2"          # "S2", "S1", or None
PLOT_KIND = "model"         # "model", "rules", or None
ROLLING_MEDIAN_POINTS = 5   # 0 or 1 disables smoothing

print("Classified raster folder:", CLASSIFIED_DIR)
print("CSV output:", CSV_PATH)
print("Figure output:", PNG_PATH)


## Rebuild the dataframe

This scans the final classified mosaics only. It does **not** rerun the classifier or require the original predictor rasters.


In [ ]:
DATE_RE = re.compile(
    r"(?P<start>\d{4}-\d{2}-\d{2})_to_(?P<end>\d{4}-\d{2}-\d{2})"
)

def describe_classified_raster(path):
    name = path.stem
    lower = name.lower()

    # Ignore probability rasters and intermediate tiles.
    if "_proba" in lower or re.search(r"_tile_\d+$", lower):
        return None

    match = DATE_RE.search(name)
    if not match:
        return None

    if name.startswith("winam_s2_"):
        sensor = "S2"
    elif name.startswith("winam_s1_"):
        sensor = "S1"
    else:
        return None

    if lower.endswith("_local_rules"):
        kind = "rules"
        method = "Paper_rules" if sensor == "S2" else "Paper_rules_SCC"
    elif "_local_" in lower:
        kind = "model"
        slug = name.split("_local_", 1)[1]
        slug = re.sub(r"_patch_cleaned$", "", slug, flags=re.IGNORECASE)
        method = f"Route_B_{slug}" if sensor == "S2" else f"Route_B_SCC_{slug}"
    else:
        return None

    return {
        "path": str(path),
        "sensor": sensor,
        "kind": kind,
        "method": method,
        "start_date": match.group("start"),
        "end_date": match.group("end"),
        "patch_cleaned": lower.endswith("_patch_cleaned"),
    }


def wh_metrics_from_geotiff(path, floating_code=2, nodata=255):
    floating_pixels = 0
    valid_pixels = 0

    with rasterio.open(path) as src:
        pixel_area_m2 = abs(src.transform.a * src.transform.e)

        for _, window in src.block_windows(1):
            arr = src.read(1, window=window)
            valid = arr != nodata
            valid_pixels += int(valid.sum())
            floating_pixels += int(np.count_nonzero(arr[valid] == floating_code))

        crs = str(src.crs)
        resolution_x = abs(src.transform.a)
        resolution_y = abs(src.transform.e)

    wh_area_m2 = floating_pixels * pixel_area_m2
    valid_area_m2 = valid_pixels * pixel_area_m2

    return {
        "wh_pixel_count": floating_pixels,
        "valid_pixel_count": valid_pixels,
        "pixel_area_m2": pixel_area_m2,
        "wh_area_m2": wh_area_m2,
        "wh_area_ha": wh_area_m2 / 10_000,
        "wh_area_km2": wh_area_m2 / 1_000_000,
        "valid_area_ha": valid_area_m2 / 10_000,
        "wh_coverage_pct": 100 * floating_pixels / valid_pixels if valid_pixels else np.nan,
        "crs": crs,
        "resolution_x": resolution_x,
        "resolution_y": resolution_y,
    }


if not CLASSIFIED_DIR.exists():
    raise FileNotFoundError(
        f"CLASSIFIED_DIR does not exist:\n{CLASSIFIED_DIR}\n"
        "Update CLASSIFIED_DIR in the configuration cell."
    )

records = []
for path in sorted(CLASSIFIED_DIR.glob("*.tif")):
    info = describe_classified_raster(path)
    if info is not None:
        records.append(info)

if not records:
    raise RuntimeError(
        "No final classified GeoTIFFs were recognised. "
        "Check CLASSIFIED_DIR and the filename pattern."
    )

catalogue = pd.DataFrame(records)

# Prefer S1 patch-cleaned outputs when both versions are present.
catalogue = (
    catalogue
    .sort_values(
        ["sensor", "kind", "method", "start_date", "end_date", "patch_cleaned"],
        ascending=[True, True, True, True, True, False],
    )
    .drop_duplicates(
        ["sensor", "kind", "method", "start_date", "end_date"],
        keep="first",
    )
    .reset_index(drop=True)
)

print(f"Recognised {len(catalogue):,} final classified rasters.")
display(catalogue.head())


In [ ]:
rows = []

for i, item in catalogue.iterrows():
    metrics = wh_metrics_from_geotiff(
        item["path"],
        floating_code=FLOATING_CLASS_CODE,
        nodata=NODATA_VALUE,
    )

    row = item.to_dict()
    row.update(metrics)
    rows.append(row)

    if (i + 1) % 25 == 0 or (i + 1) == len(catalogue):
        print(f"Processed {i + 1:,}/{len(catalogue):,} rasters")

wh_ts = pd.DataFrame(rows)

wh_ts["start_date"] = pd.to_datetime(wh_ts["start_date"])
wh_ts["end_date"] = pd.to_datetime(wh_ts["end_date"])
wh_ts["mid_date"] = (
    wh_ts["start_date"] +
    (wh_ts["end_date"] - wh_ts["start_date"]) / 2
)
wh_ts["year"] = wh_ts["mid_date"].dt.year
wh_ts["month"] = wh_ts["mid_date"].dt.month
wh_ts["year_month"] = wh_ts["mid_date"].dt.to_period("M").astype(str)

columns = [
    "sensor", "kind", "method",
    "start_date", "end_date", "mid_date", "year", "month", "year_month",
    "wh_pixel_count", "valid_pixel_count", "pixel_area_m2",
    "wh_area_m2", "wh_area_ha", "wh_area_km2",
    "valid_area_ha", "wh_coverage_pct",
    "crs", "resolution_x", "resolution_y",
    "patch_cleaned", "path",
]

wh_ts = (
    wh_ts[columns]
    .sort_values(["sensor", "kind", "method", "mid_date"])
    .reset_index(drop=True)
)

wh_ts.to_csv(CSV_PATH, index=False)

print(f"\nSaved {len(wh_ts):,} rows to:\n{CSV_PATH}")
display(wh_ts.head(10))


## Quick QA


In [ ]:
duplicate_mask = wh_ts.duplicated(
    ["sensor", "kind", "method", "start_date", "end_date"],
    keep=False,
)

print("Duplicate sensor/method/date rows:", int(duplicate_mask.sum()))
print("Date range:", wh_ts["mid_date"].min(), "to", wh_ts["mid_date"].max())

print("\nRows by sensor and method:")
display(
    wh_ts.groupby(["sensor", "kind", "method"], dropna=False)
         .size()
         .rename("n_dates")
         .reset_index()
)

print("\nWH area summary (ha):")
display(
    wh_ts.groupby(["sensor", "kind", "method"])["wh_area_ha"]
         .agg(["count", "min", "median", "mean", "max"])
         .round(2)
         .reset_index()
)


## Publication-ready time series

The default plot shows the **S2 model series**. The thin line shows individual classified dates and the thicker line is a centred five-observation rolling median used only as a visual aid.


In [ ]:
plot_df = wh_ts.copy()

if PLOT_SENSOR is not None:
    plot_df = plot_df[plot_df["sensor"].eq(PLOT_SENSOR)]

if PLOT_KIND is not None:
    plot_df = plot_df[plot_df["kind"].eq(PLOT_KIND)]

if plot_df.empty:
    raise RuntimeError(
        "The plot filters selected no rows. "
        "Change PLOT_SENSOR / PLOT_KIND in the configuration cell."
    )

fig, ax = plt.subplots(figsize=(11.5, 5.6))

for (sensor, method), group in plot_df.groupby(["sensor", "method"], sort=True):
    group = group.sort_values("mid_date")

    label = f"{sensor} | {method}"
    line, = ax.plot(
        group["mid_date"],
        group["wh_area_ha"],
        linewidth=1.2,
        alpha=0.72,
        label=label,
    )

    ax.scatter(
        group["mid_date"],
        group["wh_area_ha"],
        s=12,
        alpha=0.72,
        color=line.get_color(),
        edgecolors="none",
    )

    if ROLLING_MEDIAN_POINTS and ROLLING_MEDIAN_POINTS > 1:
        rolling = (
            group["wh_area_ha"]
            .rolling(
                ROLLING_MEDIAN_POINTS,
                center=True,
                min_periods=max(2, ROLLING_MEDIAN_POINTS // 2),
            )
            .median()
        )
        ax.plot(
            group["mid_date"],
            rolling,
            linewidth=2.4,
            color=line.get_color(),
            label=f"{sensor} | rolling median ({ROLLING_MEDIAN_POINTS} observations)",
        )

ax.set_title("Water-hyacinth extent in the Winam Gulf AOI")
ax.set_xlabel("Date")
ax.set_ylabel("Mapped water-hyacinth area (ha)")

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=(4, 7, 10)))

ax.grid(axis="y", alpha=0.22)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.margins(x=0.01)
ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure to:")
print(PNG_PATH)


### Plot percentage coverage instead

The dataframe also contains `wh_coverage_pct`, calculated as WH pixels divided by all valid classified pixels for each raster. To plot that instead, replace `wh_area_ha` with `wh_coverage_pct` in the plotting cell and change the y-axis label to `Valid classified AOI occupied by WH (%)`.
